<a href="https://colab.research.google.com/github/LAtifword/LATIF-NI/blob/main/training/LATIF_NI_Train_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LATIF NI — real LoRA training on a free Colab GPU
Run in Google Colab with **Runtime → Change runtime type → T4 GPU** (if available). This trains **Qwen2.5-0.5B**, **not Gemini Nano**. No GPU is guaranteed; do not upload confidential datasets you lack permission to process in Colab.
Prepare a UTF-8 JSONL file with **at least 20 unique examples**: one `{ "text": "..." }` per line, each text at least 30 characters. Keep a copy offline.

In [7]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU assigned. Choose GPU runtime and reconnect.'
print('CUDA:', torch.cuda.get_device_name(0))
!git clone --depth 1 --branch latif-brain https://github.com/LAtifword/latif-brain.git /content/latif-brain
%cd /content/latif-brain
!python -m pip install -q -r training/requirements.txt 'datasets>=3,<5' 'torchao>=0.16.0'

CUDA: Tesla T4
fatal: destination path '/content/latif-brain' already exists and is not an empty directory.
/content/latif-brain


In [9]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
assert len(uploaded)==1, 'Upload exactly one training JSONL file'
name = next(iter(uploaded))
Path('/content/latif-train.jsonl').write_bytes(uploaded[name])
!python training/validate_dataset.py --input /content/latif-train.jsonl --output training/data
del uploaded


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil

drive.mount("/content/drive")

src = Path("/content/latif-brain/artifacts/lora")
dst = Path("/content/drive/MyDrive/LATIF-NI/models/lora-v1")

assert (src / "adapter_model.safetensors").is_file(), \
    f"Adapter missing: {src}"

    dst.mkdir(parents=True, exist_ok=True)

    shutil.copytree(src, dst, dirs_exist_ok=True)

    print("BACKUP SUCCESSFUL:", dst)

    for p in dst.rglob("*"):
        if p.is_file():
                print(p.name, p.stat().st_size, "bytes")

In [14]:
from pathlib import Path
from google.colab import drive
import shutil

try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount status:", e)

src = Path("/content/latif-brain/artifacts/lora")
dst = Path("/content/drive/MyDrive/LATIF-NI/models/lora-v1")

assert (src / "adapter_model.safetensors").is_file(), "Training adapter not found in artifacts/lora"

dst.mkdir(parents=True, exist_ok=True)
shutil.copytree(src, dst, dirs_exist_ok=True)

print("Saved trained adapter to Google Drive:", dst)
print("Backing up evaluation report...")
eval_src = Path("/content/latif-brain/artifacts/evaluation.json")
if eval_src.is_file():
    shutil.copy(eval_src, dst / "evaluation.json")
    print("Saved evaluation.json to Google Drive")

print("Backed up files:")
for p in dst.rglob("*"):
    if p.is_file():
        print(f" - {p.name} ({p.stat().st_size} bytes)")

Drive mount status: Mountpoint must not already contain files
Saved trained adapter to Google Drive: /content/drive/MyDrive/LATIF-NI/models/lora-v1
Backing up evaluation report...
Saved evaluation.json to Google Drive
Backed up files:
 - added_tokens.json (605 bytes)
 - vocab.json (2776833 bytes)
 - tokenizer.json (11421994 bytes)
 - merges.txt (1671853 bytes)
 - tokenizer_config.json (4686 bytes)
 - README.md (5206 bytes)
 - evaluation.json (191 bytes)
 - chat_template.jinja (2507 bytes)
 - special_tokens_map.json (613 bytes)
 - adapter_model.safetensors (2175168 bytes)
 - adapter_config.json (1104 bytes)
 - scheduler.pt (1465 bytes)
 - added_tokens.json (605 bytes)
 - vocab.json (2776833 bytes)
 - rng_state.pth (14645 bytes)
 - tokenizer.json (11421994 bytes)
 - merges.txt (1671853 bytes)
 - tokenizer_config.json (4686 bytes)
 - training_args.bin (5777 bytes)
 - README.md (5206 bytes)
 - scaler.pt (1383 bytes)
 - optimizer.pt (4407947 bytes)
 - chat_template.jinja (2507 bytes)
 - spe

In [ ]:
import sys
# Install Gradio and Peft if not installed
!pip install -q gradio peft transformers accelerate bitsandbytes

import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_id = "/content/latif-brain/artifacts/lora"

print("Loading base model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Loading and applying fine-tuned LoRA adapter...")
model = PeftModel.from_pretrained(model, adapter_id)
model.eval()

def predict(message, history):
    # Prepare conversational system prompt or format
    messages = []
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})

    # Generate tokenized prompt template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    # Slice out the input prompt to get only the answer
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# Create the Chat UI
demo = gr.ChatInterface(
    fn=predict,
    title="LATIF-NI Trained Model Dashboard",
    description="Test your fine-tuned model live in the browser using this chat companion UI."
)

print("Starting dashboard UI in debug mode. Use the public URL generated below to test!")
demo.launch(share=True, debug=True)

Loading base model and tokenizer...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading and applying fine-tuned LoRA adapter...
Starting dashboard UI in debug mode. Use the public URL generated below to test!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6240e46e738e1e3d27.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1961, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gradio/utils.py", line 1083, in async_wrapper
    response = await 

In [8]:
import subprocess
from pathlib import Path

model = 'Qwen/Qwen2.5-0.5B-Instruct'

print("Starting real GPU training run and capturing output...")
result = subprocess.run([
    'python', 'training/train_lora.py',
    '--model', model,
    '--data', 'training/data/train.jsonl',
    '--output', 'artifacts/lora',
    '--steps', '100',
    '--max-length', '256'
], capture_output=True, text=True)

print("--- STDOUT ---")
print(result.stdout)
print("--- STDERR ---")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Training failed with exit code {result.returncode}")

adapter_path = Path('artifacts/lora/adapter_model.safetensors')
assert adapter_path.is_file(), 'Adapter file missing: Training failed to save weights.'
print('Training completed successfully! Adapter weight file exists at:', adapter_path)

Starting real GPU training run and capturing output...
--- STDOUT ---
trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093
{'loss': 2.2713, 'grad_norm': 0.7762365341186523, 'learning_rate': 0.000182, 'epoch': 0.22}
{'loss': 1.9216, 'grad_norm': 0.744179368019104, 'learning_rate': 0.000162, 'epoch': 0.43}
{'loss': 1.9236, 'grad_norm': 0.919496476650238, 'learning_rate': 0.000142, 'epoch': 0.65}
{'loss': 1.9833, 'grad_norm': 1.3686962127685547, 'learning_rate': 0.000122, 'epoch': 0.87}
{'loss': 1.8931, 'grad_norm': 0.840906023979187, 'learning_rate': 0.00010200000000000001, 'epoch': 1.07}
{'loss': 1.7149, 'grad_norm': 0.9846086502075195, 'learning_rate': 8.2e-05, 'epoch': 1.28}
{'loss': 1.7984, 'grad_norm': 1.1799958944320679, 'learning_rate': 6.2e-05, 'epoch': 1.5}
{'loss': 1.6624, 'grad_norm': 0.8100718259811401, 'learning_rate': 4.2e-05, 'epoch': 1.72}
{'loss': 1.8529, 'grad_norm': 1.1761912107467651, 'learning_rate': 2.2000000000000003e-05, 'epoch': 1.93}
{'loss

In [ ]:
import subprocess
from pathlib import Path

model = 'Qwen/Qwen2.5-0.5B-Instruct'

print("Evaluating the trained model on held-out validation set...")
result = subprocess.run([
    'python', 'training/evaluate_lora.py',
    '--model', model,
    '--adapter', 'artifacts/lora',
    '--data', 'training/data/validation.jsonl'
], capture_output=True, text=True)

print("--- EVALUATION OUTPUT ---")
print(result.stdout)
print(result.stderr)

eval_json = Path('artifacts/evaluation.json')
if eval_json.is_file():
    print("\nSuccessfully saved evaluation metrics to:", eval_json)
    print(eval_json.read_text())
else:
    raise RuntimeError("Evaluation failed: Output metrics file not found.")

In [12]:
import zipfile
from pathlib import Path
from google.colab import files

archive = '/content/LATIF-NI-QWEN-LORA.zip'
print("Creating ZIP archive containing the LoRA adapter weights and evaluation files...")

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    # Compress LoRA weights and configs
    lora_dir = Path('artifacts/lora')
    if lora_dir.is_dir():
        for p in lora_dir.rglob('*'):
            if p.is_file():
                z.write(p, p.relative_to(Path('artifacts').parent).as_posix())
                print(f"Added: {p.as_posix()}")

    # Compress evaluation report
    eval_file = Path('artifacts/evaluation.json')
    if eval_file.is_file():
        z.write(eval_file, eval_file.as_posix())
        print(f"Added: {eval_file.as_posix()}")

    # Compress dataset manifest
    manifest_file = Path('training/data/manifest.json')
    if manifest_file.is_file():
        z.write(manifest_file, manifest_file.as_posix())
        print(f"Added: {manifest_file.as_posix()}")

print("\nArchive ready. Initiating browser download...")
files.download(archive)
print('Download initiated! This package contains the adapter weights and evaluation json.')

Creating ZIP archive containing the LoRA adapter weights and evaluation files...
Added: artifacts/lora/added_tokens.json
Added: artifacts/lora/vocab.json
Added: artifacts/lora/tokenizer.json
Added: artifacts/lora/merges.txt
Added: artifacts/lora/tokenizer_config.json
Added: artifacts/lora/README.md
Added: artifacts/lora/chat_template.jinja
Added: artifacts/lora/special_tokens_map.json
Added: artifacts/lora/adapter_model.safetensors
Added: artifacts/lora/adapter_config.json
Added: artifacts/lora/checkpoint-50/scheduler.pt
Added: artifacts/lora/checkpoint-50/added_tokens.json
Added: artifacts/lora/checkpoint-50/vocab.json
Added: artifacts/lora/checkpoint-50/rng_state.pth
Added: artifacts/lora/checkpoint-50/tokenizer.json
Added: artifacts/lora/checkpoint-50/merges.txt
Added: artifacts/lora/checkpoint-50/tokenizer_config.json
Added: artifacts/lora/checkpoint-50/training_args.bin
Added: artifacts/lora/checkpoint-50/README.md
Added: artifacts/lora/checkpoint-50/scaler.pt
Added: artifacts/lor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated! This package contains the adapter weights and evaluation json.
